# U9 — EDA & Statistical Analysis: Lab

### Real-world brief: reducing customer churn at an e-commerce company

You've just joined the data team. The business wants to **predict and reduce churn** (customers who stop buying). Before anyone builds a model, your job is to *explore the data*: understand its shape, surface data-quality problems, and find which signals relate to churn.

**Resources provided with this lab:** `ecommerce_customers.csv` (one row per customer) and `transactions.xlsx` (one row per order). Keep them in the same folder as this notebook (in Colab, upload them via the Files panel).

_Day 4–5 · Phase C — Data Engineering & Preparation._

#objectives

By the end of this lab you will be able to:

Profile a real dataset and quantify its data-quality issues

Analyse single variables (numeric & categorical) and read their distributions

Study relationships with scatter plots, group comparisons and a correlation heatmap

Use groupby / pivot to find which customer segments churn most

Spot the classic EDA red flags: skew, outliers, class imbalance and missingness

#how to use this lab

Worked demo cells teach the pattern — run them and read the comments.

LAB EXERCISE cells (marked 🧪) are framed as questions the business would actually ask. Replace each `# YOUR CODE HERE` with working code.

Run cells with **Shift + Enter**, top to bottom.

In [ ]:
# === SETUP: load the provided data files (regenerate them if missing) ===
import os
import numpy as np
import pandas as pd

def build_datasets(csv_path='ecommerce_customers.csv',
                   xlsx_path='transactions.xlsx', seed=42, verbose=False):
    """Generate a realistic e-commerce customer + transactions dataset.

    Baked-in realism for EDA / feature engineering practice:
      - right-skewed monetary columns (total_spend) with a few 'whale' outliers
      - class imbalance in is_churned (~20-30%)
      - real signal: churn depends on recency, order count and support tickets
      - missing values in age / gender / city
      - a high-cardinality 'city' column (long tail of rare cities)
      - customers with zero orders (no last_order_date) -> dormant
    The Excel file is order-level and is CONSISTENT with the customer table
    (num_orders / total_spend / last_order_date are derived from it).
    """
    rng = np.random.default_rng(seed)
    N = 2500
    start = pd.Timestamp('2021-01-01')
    end = pd.Timestamp('2024-06-30')
    horizon = (end - start).days

    cust = np.array([f'CUST{i+1:05d}' for i in range(N)])
    signup_off = rng.integers(0, horizon - 60, N)
    signup = start + pd.to_timedelta(signup_off, unit='D')

    # order counts: overdispersed (gamma-poisson), some customers have zero
    lam = rng.gamma(2.0, 1.6, N)
    num_orders = rng.poisson(lam)

    # ---- order-level transactions (vectorised) ----
    counts = num_orders
    tot = int(counts.sum())
    cust_rep = np.repeat(cust, counts)
    signup_rep = np.repeat(signup_off, counts)
    span = np.maximum(horizon - signup_off, 1)
    span_rep = np.repeat(span, counts)
    off = (rng.random(tot) * span_rep).astype(int)
    tx_off = signup_rep + off
    tx_date = start + pd.to_timedelta(tx_off, unit='D')
    amount = rng.lognormal(3.2, 0.8, tot).round(2)        # right-skewed (~tens of currency)
    category = rng.choice(['Electronics', 'Fashion', 'Grocery', 'Home', 'Books'],
                          tot, p=[.20, .30, .25, .15, .10])
    tx = pd.DataFrame({'customer_id': cust_rep, 'order_date': tx_date,
                       'amount': amount, 'category': category}).sort_values(
        ['customer_id', 'order_date']).reset_index(drop=True)

    # ---- aggregate transactions -> customer level ----
    agg = tx.groupby('customer_id').agg(
        total_spend=('amount', 'sum'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    ).reset_index()

    df = pd.DataFrame({'customer_id': cust, 'signup_date': signup,
                       'num_orders': num_orders})
    df = df.merge(agg, on='customer_id', how='left')
    df['total_spend'] = df['total_spend'].fillna(0).round(2)

    # ---- demographics & account attributes ----
    df['age'] = np.clip(rng.normal(38, 12, N), 18, 82).round().astype(int)
    df['gender'] = rng.choice(['M', 'F', 'Other'], N, p=[.48, .48, .04])

    majors = ['Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
    rare = ['Jaipur', 'Surat', 'Indore', 'Bhopal', 'Patna', 'Nagpur',
            'Kochi', 'Coimbatore', 'Visakhapatnam', 'Lucknow']
    pool = majors + rare
    w = np.array([.17, .15, .14, .12, .10, .08, .06] + [.013] * 10)
    w = w / w.sum()
    df['city'] = rng.choice(pool, N, p=w)

    df['plan'] = rng.choice(['Basic', 'Standard', 'Premium'], N, p=[.50, .35, .15])
    df['device'] = rng.choice(['Mobile', 'Desktop', 'Tablet'], N, p=[.60, .32, .08])
    df['payment_method'] = rng.choice(['Card', 'UPI', 'Wallet', 'NetBanking'],
                                      N, p=[.40, .35, .15, .10])
    df['support_tickets'] = rng.poisson(0.6, N)
    df['email_opt_in'] = rng.choice([0, 1], N, p=[.35, .65])

    # ---- churn target with real signal (recency / orders / tickets) ----
    last = pd.to_datetime(df['last_order'])
    days_since = (end - last).dt.days
    days_since_filled = days_since.fillna(horizon).to_numpy()
    z = (-2.75
         + 0.0019 * days_since_filled
         + 0.30 * df['support_tickets'].to_numpy()
         - 0.05 * df['num_orders'].to_numpy()
         + 0.70 * (df['num_orders'].to_numpy() == 0))
    p = 1 / (1 + np.exp(-z))
    df['is_churned'] = (rng.random(N) < p).astype(int)

    # ---- format dates as ISO strings (NaT -> <NA>) ----
    df = df.rename(columns={'first_order': 'first_order_date',
                            'last_order': 'last_order_date'})
    for c in ['signup_date', 'first_order_date', 'last_order_date']:
        df[c] = pd.to_datetime(df[c]).dt.date.astype('string')

    df = df[['customer_id', 'signup_date', 'first_order_date', 'last_order_date',
             'age', 'gender', 'city', 'plan', 'device', 'payment_method',
             'num_orders', 'total_spend', 'support_tickets', 'email_opt_in',
             'is_churned']]

    # ---- inject missing values AFTER computing the target ----
    def punch(col, frac):
        idx = rng.choice(N, int(frac * N), replace=False)
        df.loc[idx, col] = np.nan
    punch('age', 0.07)
    punch('gender', 0.04)
    punch('city', 0.02)

    # ---- write files ----
    df.to_csv(csv_path, index=False)
    tx_out = tx.copy()
    tx_out['order_date'] = pd.to_datetime(tx_out['order_date']).dt.date.astype('string')
    tx_out.to_excel(xlsx_path, index=False)

    if verbose:
        print('customers:', df.shape, '| transactions:', tx_out.shape)
        print('churn rate:', round(df["is_churned"].mean(), 3))
        print('total_spend skew:', round(df["total_spend"].skew(), 2))
        print('missing age:', int(df["age"].isna().sum()),
              '| missing city:', int(df["city"].isna().sum()))
        print('zero-order customers:', int((df["num_orders"] == 0).sum()))
        print('distinct cities:', df["city"].nunique())
    return df, tx_out

if not (os.path.exists('ecommerce_customers.csv') and os.path.exists('transactions.xlsx')):
    build_datasets()   # creates the two resource files locally
    print('Generated dataset files.')
else:
    print('Found the provided dataset files.')

In [ ]:
# Load the customer table (parse the date columns as real datetimes)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

df = pd.read_csv('ecommerce_customers.csv',
                 parse_dates=['signup_date', 'first_order_date', 'last_order_date'])
print('Loaded', df.shape[0], 'customers x', df.shape[1], 'columns')
df.head()

#1. First look — profile the dataset

In [ ]:
# -----------------------------------------------------------
# 🔹 1A. STRUCTURE: what are we working with?
# -----------------------------------------------------------
df.info()   # column types + non-null counts

In [ ]:
# -----------------------------------------------------------
# 🔹 1B. QUALITY SNAPSHOT: missingness + the target balance
# -----------------------------------------------------------
print('Missing values (%):')
print((df.isna().mean() * 100).round(1).sort_values(ascending=False).head(6))
print('\nChurn rate (target):', round(df['is_churned'].mean(), 3))
print('Customers with zero orders:', int((df['num_orders'] == 0).sum()))

#### 🧪 LAB EXERCISE 1 — Data-quality audit

Management asks: *"Can we trust this data?"* Produce a quick audit:
1. Confirm `customer_id` is unique (no duplicate customers).
2. Show the numeric summary with `df.describe()` and eyeball for anything odd (e.g. a min of 0).
3. In a comment, list **three** data-quality issues you can already see.

In [ ]:
# 1. is customer_id unique?
# YOUR CODE HERE

# 2. numeric summary
# YOUR CODE HERE

# 3. Three issues I can see: ...   (comment)

#2. Univariate analysis — numeric variables

In [ ]:
# -----------------------------------------------------------
# 🔹 2A. DISTRIBUTION OF SPEND  (note the long right tail)
# -----------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(df['total_spend'], bins=40, ax=ax[0], color='#2D6A9F')
ax[0].set_title('total_spend — right-skewed')
sns.histplot(df['num_orders'], bins=30, ax=ax[1], color='#3AAFA9')
ax[1].set_title('num_orders')
plt.tight_layout(); plt.show()

#### 🧪 LAB EXERCISE 2 — Customer age profile

Marketing wants to know the age profile of the customer base.
1. Plot a histogram of `age` (drop missing values first with `.dropna()`).
2. In a comment, describe the shape (roughly symmetric? skewed? any odd values?).

In [ ]:
# 1. histogram of age (drop NaN)
# YOUR CODE HERE

# 2. Shape description: ...   (comment)

#3. Summary statistics & skew

In [ ]:
# -----------------------------------------------------------
# 🔹 3A. MEAN vs MEDIAN ON A SKEWED COLUMN
# -----------------------------------------------------------
# When data is right-skewed, the mean is pulled up by big spenders.
spend = df['total_spend']
print('mean  :', round(spend.mean(), 1))
print('median:', round(spend.median(), 1))
print('skew  :', round(spend.skew(), 2), ' (> 0 means right-skewed)')
print('\nThe mean > median gap is the skew talking.')

#### 🧪 LAB EXERCISE 3 — Which average should finance report?

1. Compute the **skew** of `num_orders` and `total_spend`.
2. For each, decide whether **mean or median** better represents a 'typical' customer.
3. Write your recommendation in a comment.

In [ ]:
# 1. skew of the two columns
# YOUR CODE HERE

# 2 & 3. Which average for each, and why: ...   (comment)

#4. Univariate analysis — categorical variables

In [ ]:
# -----------------------------------------------------------
# 🔹 4A. CATEGORY FREQUENCIES
# -----------------------------------------------------------
print('Plan mix:')
print(df['plan'].value_counts(normalize=True).round(3))
print('\nTop cities:')
print(df['city'].value_counts().head(5))

fig, ax = plt.subplots(figsize=(6, 3))
df['plan'].value_counts().plot(kind='bar', ax=ax, color='#3AAFA9')
ax.set_title('Customers per plan'); plt.tight_layout(); plt.show()

#### 🧪 LAB EXERCISE 4 — Is the target imbalanced?

A churn model can be misled by class imbalance, so quantify it.
1. Print `is_churned.value_counts(normalize=True)` — the churn vs retained split.
2. Compute the **imbalance ratio** (retained ÷ churned).
3. Plot a bar chart of `device` and note the dominant category in a comment.

In [ ]:
# 1. churn vs retained proportions
# YOUR CODE HERE

# 2. imbalance ratio (retained / churned)
# YOUR CODE HERE

# 3. bar chart of device + comment on the dominant category
# YOUR CODE HERE

#5. Bivariate analysis — numeric vs numeric

In [ ]:
# -----------------------------------------------------------
# 🔹 5A. DO SPEND AND ORDER COUNT MOVE TOGETHER?
# -----------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(df['num_orders'], df['total_spend'], alpha=0.3, color='#2D6A9F', s=14)
ax.set_xlabel('num_orders'); ax.set_ylabel('total_spend')
ax.set_title('Spend vs orders'); plt.tight_layout(); plt.show()

print('correlation:', round(df['num_orders'].corr(df['total_spend']), 3))

#### 🧪 LAB EXERCISE 5 — Does age relate to spend?

1. Make a scatter plot of `age` (x) vs `total_spend` (y).
2. Compute their correlation.
3. In a comment, say whether there's a meaningful relationship — and recall *correlation ≠ causation*.

In [ ]:
# 1. scatter age vs total_spend
# YOUR CODE HERE

# 2. correlation
# YOUR CODE HERE

# 3. Interpretation: ...   (comment)

#6. Bivariate analysis — numeric vs categorical (churned vs retained)

In [ ]:
# -----------------------------------------------------------
# 🔹 6A. HOW DO CHURNERS DIFFER?  (group means)
# -----------------------------------------------------------
cols = ['num_orders', 'total_spend', 'support_tickets']
print(df.groupby('is_churned')[cols].mean().round(1))

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=df, x='is_churned', y='num_orders', ax=ax,
            palette=['#3AAFA9', '#F4B942'])
ax.set_title('Orders: retained (0) vs churned (1)')
plt.tight_layout(); plt.show()

#### 🧪 LAB EXERCISE 6 — Do churners raise more tickets?

1. Group by `is_churned` and compare the mean `support_tickets`.
2. Draw a box plot of `support_tickets` split by `is_churned`.
3. In a comment, state whether support tickets look like a churn warning sign.

In [ ]:
# 1. mean support_tickets by churn
# YOUR CODE HERE

# 2. box plot of support_tickets by is_churned
# YOUR CODE HERE

# 3. Is it a warning sign? ...   (comment)

#7. Multivariate — the correlation heatmap

In [ ]:
# -----------------------------------------------------------
# 🔹 7A. WHAT CORRELATES WITH CHURN?
# -----------------------------------------------------------
num = ['age', 'num_orders', 'total_spend', 'support_tickets',
       'email_opt_in', 'is_churned']
corr = df[num].corr()

fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, cbar_kws={'shrink': .8})
ax.set_title('Correlation matrix'); plt.tight_layout(); plt.show()

print('Correlation with churn (sorted):')
print(corr['is_churned'].drop('is_churned').sort_values())

#### 🧪 LAB EXERCISE 7 — Name the churn drivers

1. From the correlation matrix, print the two features **most correlated** with `is_churned` (by absolute value).
2. In a comment, say which direction each pushes churn (more orders → less churn, etc.).

In [ ]:
# 1. top-2 absolute correlations with churn
#    hint: corr['is_churned'].drop('is_churned').abs().sort_values(ascending=False).head(2)
# YOUR CODE HERE

# 2. Direction of each driver: ...   (comment)

#8. Grouping & aggregation — which segments churn?

In [ ]:
# -----------------------------------------------------------
# 🔹 8A. CHURN RATE BY SEGMENT
# -----------------------------------------------------------
print('Churn rate by plan:')
print(df.groupby('plan')['is_churned'].mean().round(3).sort_values(ascending=False))

print('\nChurn rate by device x plan:')
print(df.pivot_table(values='is_churned', index='plan',
                     columns='device', aggfunc='mean').round(3))

#### 🧪 LAB EXERCISE 8 — Where is churn worst?

1. Compute the churn rate by `payment_method`.
2. Build a `pivot_table` of churn rate with `plan` as rows and `email_opt_in` as columns.
3. In a comment, name the single worst-churning segment you found.

In [ ]:
# 1. churn rate by payment_method
# YOUR CODE HERE

# 2. pivot: churn rate by plan x email_opt_in
# YOUR CODE HERE

# 3. Worst-churning segment: ...   (comment)

#9. EDA red flags — the things that bite you later

In [ ]:
# -----------------------------------------------------------
# 🔹 9A. OUTLIERS IN SPEND (IQR rule)
# -----------------------------------------------------------
q1, q3 = df['total_spend'].quantile([0.25, 0.75])
iqr = q3 - q1
high = q3 + 1.5 * iqr
outliers = df[df['total_spend'] > high]
print(f'IQR upper bound: {high:.0f}')
print('Whale customers above it:', len(outliers))
print(outliers[['customer_id', 'num_orders', 'total_spend']]
      .sort_values('total_spend', ascending=False).head())

#### 🧪 LAB EXERCISE 9 — Red-flag report

Summarise the modelling risks you found, in code:
1. **Skew** — print the skew of `total_spend` (already know it's > 1).
2. **Imbalance** — print the churn rate (the rare class).
3. **Missingness** — print the column with the highest missing percentage.
4. In a comment, suggest one fix for each flag (these become your U10 to-do list).

In [ ]:
# 1. skew of total_spend
# YOUR CODE HERE

# 2. churn rate (rare-class share)
# YOUR CODE HERE

# 3. column with the most missing values
# YOUR CODE HERE

# 4. One fix per flag: skew -> ... ; imbalance -> ... ; missing -> ...   (comment)

#📘 Summary — what your EDA found

| Question | Tool | Typical finding here |
| -------- | ---- | -------------------- |
| What's in the data? | `info` · `describe` · `isna` | missing age/city, dormant customers |
| Distribution shape? | `histplot` · `.skew()` | `total_spend` is right-skewed |
| Target balance? | `value_counts(normalize=True)` | churn is the minority class (~17%) |
| Relationships? | scatter · `.corr()` · heatmap | orders & recency relate to churn |
| Which segments? | `groupby` · `pivot_table` | churn varies by plan / device |
| Red flags? | IQR · skew · missingness | skew, whales, imbalance, gaps |

**Every finding becomes an action in U10:** right-skew → log transform · whales → cap · high-cardinality city → group rare · recency signal → engineer a `days_since_last_order` feature.

**Next — U10 Feature Engineering (Part 1):** turn these insights into features for a churn model.